# List Comprehensions and Vectorized Neural Network Activations

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

List comprehensions provide a concise syntactic construct derived from mathematical set-builder notation:

$$\{ f(x) \mid x \in S \land P(x) \}$$

In Python, comprehensions are not merely syntactic sugar: they compile directly into specialized C-level bytecode instructions (`LIST_APPEND`) that bypass the repeated method lookup overhead of `list.append()`, executing significantly faster.

### Learning Objectives
1. Master comprehension mapping and filtering semantics.
2. Disambiguate ternary transformation (`val_a if cond else val_b`) vs. predicate filtering (`if cond`).
3. Vectorize non-linear neural network activations (Sigmoid, ReLU, Leaky ReLU, Step) across feature vectors.
4. Work with multidimensional comprehensions for matrix flattening and transposition.
5. Implement a feature standardization and activation layer pipeline.

In [ ]:
# Basic Comprehension Mechanics
raw_data: list[int] = [6, 5, 7, 1, 9, 2]

# Mapping: [f(x) for x in data]
squared_data: list[int] = [x * x for x in raw_data]
print(f"Squared: {squared_data}")
assert squared_data == [36, 25, 49, 1, 81, 4]

# Filtering: [x for x in data if condition(x)]
odd_numbers: list[int] = [x for x in raw_data if x % 2 != 0]
print(f"Odds:    {odd_numbers}")
assert odd_numbers == [5, 7, 1, 9]
print("Basic comprehension assertions passed.")

## Disambiguating Ternary Mapping vs. Predicate Filtering

A frequent point of confusion in Python syntax arises between conditional transformation and conditional filtering:

1. **Ternary Mapping (Output Transformation):**
   ```python
   [x if x > 0 else 0 for x in data]
   ```
   - Placed **before** `for x in data`.
   - Must provide both `if` and `else`.
   - Preserves the total length of the sequence, transforming elements conditionally.

2. **Predicate Filtering (Subset Selection):**
   ```python
   [x for x in data if x > 0]
   ```
   - Placed **after** `for x in data`.
   - Does **not** include an `else` clause.
   - Contracts the sequence, retaining only items where the condition evaluates to `True`.

In [ ]:
# Disambiguation demonstration
mixed_values: list[int] = [1, 5, -4, 3, -2]

# Pattern A: Ternary Mapping (length preserved)
zero_clamped: list[int] = [x if x > 0 else 0 for x in mixed_values]
print(f"Zero-clamped: {zero_clamped}")
assert len(zero_clamped) == len(mixed_values)
assert zero_clamped == [1, 5, 0, 3, 0]

# Pattern B: Predicate Filtering (length reduced)
positive_only: list[int] = [x for x in mixed_values if x > 0]
print(f"Positive only: {positive_only}")
assert len(positive_only) == 3
assert positive_only == [1, 5, 3]
print("Disambiguation assertions validated.")

## Vectorized Neural Network Activations

Machine learning architectures rely on non-linear activation functions applied element-wise across pre-activation tensors $\mathbf{z}$:

1. **Sigmoid:**
   $$\sigma(z) = \frac{1}{1 + e^{-z}} \in (0, 1)$$
2. **Rectified Linear Unit (ReLU):**
   $$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$
3. **Leaky ReLU (with slope $\alpha = 0.01$):**
   $$\text{LeakyReLU}(z) = \begin{cases} z & \text{if } z > 0 \\ \alpha z & \text{if } z \le 0 \end{cases}$$
4. **Binary Step (Heaviside):**
   $$H(z) = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$

In [ ]:
import math


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def vectorize_sigmoid(inputs: list[float]) -> list[float]:
    return [sigmoid(x) for x in inputs]


def vectorize_relu(inputs: list[float]) -> list[float]:
    return [x if x > 0.0 else 0.0 for x in inputs]


def vectorize_leaky_relu(inputs: list[float], alpha: float = 0.01) -> list[float]:
    return [x if x > 0.0 else alpha * x for x in inputs]


def vectorize_step(inputs: list[float]) -> list[float]:
    return [1.0 if x >= 0.0 else 0.0 for x in inputs]


# Verification
test_z: list[float] = [-3.0, -1.0, 0.0, 2.0, 5.0]

sig_out = vectorize_sigmoid(test_z)
relu_out = vectorize_relu(test_z)
lrelu_out = vectorize_leaky_relu(test_z)
step_out = vectorize_step(test_z)

assert abs(sig_out[2] - 0.5) < 1e-9
assert relu_out == [0.0, 0.0, 0.0, 2.0, 5.0]
assert lrelu_out[0] == -0.03
assert step_out == [0.0, 0.0, 1.0, 1.0, 1.0]
print("All vectorized activation functions validated.")

## Practical Lab Exercise: Matrix Transposition and Standardization Layer

Implement:
1. `matrix_transpose(matrix: list[list[float]]) -> list[list[float]]`: Transposes a $2D$ matrix using nested list comprehensions.
2. `standardize_and_activate(features: list[float], activation: str = 'relu') -> list[float]`: Standardizes a numeric feature vector (zero mean, unit variance) and applies the requested non-linear activation.

In [ ]:
def matrix_transpose(matrix: list[list[float]]) -> list[list[float]]:
    if not matrix or not matrix[0]:
        return []
    num_rows = len(matrix)
    num_cols = len(matrix[0])
    return [[matrix[r][c] for r in range(num_rows)] for c in range(num_cols)]


def standardize_and_activate(features: list[float], activation: str = "relu") -> list[float]:
    n = len(features)
    if n == 0:
        return []
    mean = sum(features) / n
    variance = sum((x - mean) ** 2 for x in features) / n
    std = math.sqrt(variance) if variance > 1e-12 else 1.0

    standardized = [(x - mean) / std for x in features]
    if activation == "relu":
        return [max(0.0, z) for z in standardized]
    elif activation == "sigmoid":
        return [1.0 / (1.0 + math.exp(-z)) for z in standardized]
    raise ValueError(f"Unsupported activation: {activation}")


# Test matrix transpose
mat = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
mat_t = matrix_transpose(mat)
assert mat_t == [[1.0, 4.0], [2.0, 5.0], [3.0, 6.0]]

# Test standardize + relu
feats = [10.0, 20.0, 30.0, 40.0, 50.0]
activated = standardize_and_activate(feats, "relu")
assert len(activated) == 5
assert activated[0] == 0.0  # Lowest score standardized to negative, clamped to 0
assert activated[-1] > 0.0
print("Matrix and standardization exercises passed successfully!")